# Aggregate Scores Pipeline

Reads per-branch coverage details and aggregates them into
final tract-level scores: raw score, percent covered by
at least one library, and number of branches reaching each tract.

Processes all modes and travel times in the coverage results.

In [ ]:
import os
import kfp
from kfp import dsl, kubernetes
from kfp.dsl import Output, Dataset

## Pipeline Component

In [ ]:
RUNTIME_IMAGE = "image-registry.openshift-image-registry.svc:5000/redhat-ods-applications/runtime-datascience:datascience"


@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "shapely", "pandas", "pyarrow", "pyproj"],
    pip_index_urls=["https://pypi.org/simple"],
)
def aggregate_scores(
    aggregate_out: Output[Dataset],
):
    import io
    import os
    import logging
    import pandas as pd
    import boto3
    from shapely import wkt
    from shapely.ops import unary_union
    from pyproj import Geod

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("aggregate_scores")

    geod = Geod(ellps="WGS84")

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    # Load coverage details
    obj = s3.get_object(Bucket=bucket, Key="results/coverage_details.parquet")
    coverage = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    logger.info(f"Loaded {len(coverage)} coverage records")

    if coverage.empty:
        logger.warning("No coverage data to aggregate")
        pd.DataFrame().to_parquet(aggregate_out.path, index=False)
        return

    # Load tract geometries for area computation
    obj = s3.get_object(Bucket=bucket, Key="clean/population_demographics.csv")
    tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))

    tract_area_map = {}
    for _, row in tracts.iterrows():
        geom = wkt.loads(row["geometry"])
        area, _ = geod.geometry_area_perimeter(geom)
        tract_area_map[row["geoid"]] = abs(area)

    # Aggregate per (geoid, mode, travel_time)
    results = []
    groups = coverage.groupby(["geoid", "mode", "travel_time"])
    total_groups = len(groups)

    for i, ((geoid, mode, tt), group) in enumerate(groups):
        raw_score = group["overlap_proportion"].sum()
        num_branches = len(group)

        intersection_geoms = [wkt.loads(w) for w in group["intersection_wkt"]]
        merged = unary_union(intersection_geoms)
        merged_area, _ = geod.geometry_area_perimeter(merged)
        merged_area = abs(merged_area)
        tract_area = tract_area_map.get(geoid, 1)
        percent_covered = merged_area / tract_area if tract_area > 0 else 0

        results.append({
            "geoid": geoid,
            "mode": mode,
            "travel_time": tt,
            "raw_score": raw_score,
            "percent_covered": min(percent_covered, 1.0),
            "num_branches": num_branches,
        })

        if (i + 1) % 500 == 0:
            logger.info(f"Aggregated {i + 1}/{total_groups} groups")

    result_df = pd.DataFrame(results)

    # Write parquet to S3
    buf = io.BytesIO()
    result_df.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/coverage_scores.parquet", Body=buf.getvalue())

    # Write CSV for easy viewing
    buf = io.BytesIO()
    result_df.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/coverage_scores.csv", Body=buf.getvalue())

    result_df.to_parquet(aggregate_out.path, index=False)
    logger.info(
        f"Aggregated {len(result_df)} tract scores across "
        f"{result_df['mode'].nunique()} modes. "
        f"Written to s3://{bucket}/results/coverage_scores.parquet"
    )

## Pipeline Definition

In [ ]:
@dsl.pipeline(
    name="cpl-aggregate-scores",
    description="Aggregate coverage details into final tract-level scores",
)
def aggregate_scores_pipeline():
    task = aggregate_scores()
    kubernetes.use_secret_as_env(
        task,
        secret_name="cpl-s3",
        secret_key_to_env={
            "AWS_ACCESS_KEY_ID": "AWS_ACCESS_KEY_ID",
            "AWS_SECRET_ACCESS_KEY": "AWS_SECRET_ACCESS_KEY",
            "AWS_S3_ENDPOINT": "AWS_S3_ENDPOINT",
            "AWS_S3_BUCKET": "AWS_S3_BUCKET",
        },
    )

## Submit

In [ ]:
import subprocess
from kfp.compiler import Compiler
from kfp_server_api.exceptions import ApiException

token = subprocess.check_output("oc whoami -t", shell=True, text=True).strip()
pipeline_url = os.environ.get("PIPELINES_URL")

client = kfp.Client(
    host=pipeline_url,
    existing_token=token,
    verify_ssl=False,
)

Compiler().compile(aggregate_scores_pipeline, "aggregate_scores_pipeline.yaml")

try:
    client.upload_pipeline("aggregate_scores_pipeline.yaml", pipeline_name="cpl-aggregate-scores")
    print("Pipeline definition registered.")
except ApiException as e:
    if e.status == 409:
        print("Pipeline definition already exists.")
    else:
        raise

run = client.create_run_from_pipeline_func(
    aggregate_scores_pipeline,
    experiment_name="cpl-coverage",
    run_name="aggregate-scores-run",
    enable_caching=False,
)
print(f"Pipeline run submitted: {run.run_id}")